# 🤖 [2차시] 인공지능 귀 만들기: Whisper AI와 나만의 스마트 음성 비서
> **수업 대상**: 인공지능 음성인식(STT)을 직접 실습하고 나만의 스마트 비서를 만들고 싶은 중학생 및 초심자  
> **준비물**: 구글 코랩(Google Colab), 마이크(노트북 기본 마이크 가능)  
> **핵심 키워드**: STT(Speech-to-Text), OpenAI Whisper, 음성 비서 프로젝트, 발음 클리닉
---
### 💡 오늘 우리는 무엇을 만들까요?
1. 🎤 **웹 브라우저에서 내 목소리 3초 녹음하기** (코랩 전용 마이크 녹음기)
2. 🧠 **세계 최강 음성인식 AI 'Whisper' 탑재하기** (단 3줄로 한국어 완벽 인식!)
3. 🎮 **[프로젝트 1] AI 발음 정확도 측정 게임** (간장 공장 공장장 챌린지)
4. 🏫 **[프로젝트 2] 우리 반 전용 '음성 명령 스마트 비서'** ("오늘 급식 뭐야?", "시간표 알려줘" 음성 제어!)


## 1. 📦 [1분 설치] 최신 음성인식 AI 'Whisper'와 텍스트-음성 변환기(gTTS) 설치
세계 최고의 인공지능 연구소 OpenAI가 개발한 **Whisper(위스퍼)**는 전 세계 수십만 시간의 다국어 음성을 학습하여, 사투리나 배경 소음이 있어도 사람보다 말을 더 정확하게 알아듣습니다.


In [ ]:
# Whisper AI 및 오디오 처리 라이브러리 설치 (약 30초 소요)
!pip install -q openai-whisper
!pip install -q gTTS

import whisper
import os
from gtts import gTTS
import IPython.display as ipd

print("⏳ Whisper AI 인공지능 모델(base)을 불러오는 중입니다...")
# 가볍고 빠른 base 모델 탑재 (GPU/CPU 모두 원활히 동작)
model = whisper.load_model("base")
print("🎉 AI의 귀(Whisper)가 활성화되었습니다!")


---
## 2. 🎤 [신기한 실습] 구글 코랩에서 내 목소리 직접 녹음하기!
아래 코드를 실행하면 노트북 화면에 **[🎙️ 녹음 시작]** 버튼이 나타납니다.  
버튼을 누르고 **"안녕하세요! 인공지능 실습 학생입니다."**라고 또박또박 말해보세요! (3~5초 후 정지)


In [ ]:
# 코랩 환경에서 브라우저 마이크를 사용하는 자바스크립트 녹음 위젯
from google.colab import output
from base64 import b64decode

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.stop()
  await sleep(200)
  blob = new Blob(chunks)
  resolve(await b2text(blob))
})
"""

def record_my_voice(sec=4, filename="my_voice.wav"):
    print(f"🎙️ 마이크 권한을 허용하고, {sec}초 동안 말씀하세요! (녹음 시작...)")
    display(ipd.Javascript(RECORD_JS))
    js_result = output.eval_js(f'record({sec * 1000})')
    audio_bytes = b64decode(js_result.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(audio_bytes)
    print(f"✅ 녹음 완료! '{filename}' 파일로 저장되었습니다.")
    return filename

# 마이크가 없거나 로컬 환경인 경우를 위해 가상 음성 샘플 생성기도 함께 준비했습니다.
def create_sample_audio(text="안녕하세요 학생 여러분 반갑습니다", filename="sample.wav"):
    tts = gTTS(text=text, lang='ko')
    tts.save(filename)
    return filename

# 👇 직접 마이크로 녹음하려면 아래 주석을 풀고 실행하세요!
# record_my_voice(sec=4, filename="user_speech.wav")

# 마이크가 없는 환경에서도 실습할 수 있도록 기본 샘플 음성을 생성합니다.
sample_file = create_sample_audio("안녕하세요 학생 여러분 반갑습니다 오늘의 인공지능 수업을 시작합니다", "test_speech.wav")
print("🔊 생성된 음성을 들어보세요:")
ipd.Audio(sample_file)


---
## 3. 🧠 [핵심 실습] Whisper AI로 소리를 텍스트로 변환하기(STT)!
AI에게 방금 녹음한 오디오 파일을 건네주면, 단 2~3줄의 코드로 글자로 번역해냅니다.


In [ ]:
# AI에게 음성 파일 전달
target_audio = "test_speech.wav"  # 직접 마이크 녹음했다면 "user_speech.wav"로 변경해보세요!

print("🤖 AI가 음성을 분석하고 글자로 받아 적는 중...")
result = model.transcribe(target_audio, language="ko")

print("
" + "="*50)
print(f"📝 [AI가 인식한 텍스트]: "{result['text'].strip()}"")
print("="*50)


---
## 4. 🎮 [미니 프로젝트 1] AI 발음 클리닉: "간장 공장 공장장" 챌린지!
어려운 발음을 AI가 얼마나 정확하게 알아듣는지 점수를 매겨보는 게임입니다!


In [ ]:
# 발음 클리닉 게임 함수
def pronunciation_game(target_sentence, user_audio_path):
    print(f"🎯 도전 문장: "{target_sentence}"")
    result = model.transcribe(user_audio_path, language="ko")
    recognized = result['text'].strip()
    print(f"🗣️ AI가 들은 문장: "{recognized}"")
    
    # 간단한 단어 일치율 채점
    target_words = set(target_sentence.split())
    rec_words = set(recognized.split())
    correct = len(target_words & rec_words)
    score = int((correct / max(len(target_words), 1)) * 100)
    
    print(f"
📊 발음 정확도 점수: {score}점 / 100점")
    if score >= 80:
        print("🏆 대단해요! 아나운서 뺨치는 완벽한 발음입니다! 🎉")
    elif score >= 50:
        print("👍 좋아요! 조금만 더 또박또박 발음해보세요! 😊")
    else:
        print("💡 AI가 헷갈려해요! 천천히 한 글자씩 다시 도전해보세요! 🚀")

# 샘플 테스트 실행
sample_test = create_sample_audio("간장 공장 공장장은 강 공장장이다", "tongue_twister.wav")
pronunciation_game("간장 공장 공장장은 강 공장장이다", sample_test)


---
## 5. 👥 [핵심 실습] 2명의 학생 목소리 유사도 대결 (코사인 유사도로 소리 지문 비교하기!)
> **학습 목표**: 2명의 학생이 마이크로 말했을 때, 소리의 파형(Waveform)과 스펙트럼에서 **5대 수학적 특징(RMS, ZCR, 저/중/고 대역 에너지)**을 추출하고,
> **코사인 유사도(Cosine Similarity)**를 계산하여 두 사람의 목소리가 몇 %나 일치하는지 과학적으로 분석합니다!

### 💡 소리 지문의 5가지 수학적 특징:
1. **RMS (Root Mean Square)**: 소리의 실효 음량 크기
2. **ZCR (Zero Crossing Rate)**: 소리가 0을 교차하는 횟수 (거칠기, 자음/마찰음 지표)
3. **Low Band (0~300Hz)**: 목소리의 기본 진동수(피치) 에너지
4. **Mid Band (300~3,000Hz)**: 모음의 포만트(발음 정보) 에너지
5. **High Band (3,000~8,000Hz)**: 자음의 치찰음 및 고주파 성분

In [ ]:
# [실습] 2명의 학생 음성 특징 추출 및 코사인 유사도(일치율 %) 계산기
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf

def make_feature_from_wave(y, sr=16000):
    """소리 파형 y에서 5차원 정규화 지문 벡터를 추출합니다."""
    if y.ndim > 1:
        y = y.mean(axis=1)  # 스테레오 -> 모노
    y = y.astype(np.float32)
    peak = np.max(np.abs(y)) + 1e-12
    y = (y / peak) * 0.98   # 정규화

    # 고속 푸리에 변환 (FFT)
    N = len(y)
    if N < 2048:
        y = np.pad(y, (0, 2048 - N))
        N = len(y)
    win = np.hanning(N).astype(np.float32)
    X = np.fft.rfft(y * win)
    freqs = np.fft.rfftfreq(N, 1 / sr)
    mag = np.abs(X)

    # 5대 수학적 특징
    rms = float(np.sqrt(np.mean(y**2) + 1e-12))
    zcr = float(np.mean(np.abs(np.diff(np.sign(y))) > 0))

    def band_energy(f_low, f_high):
        idx = (freqs >= f_low) & (freqs < f_high)
        return float(np.sum(mag[idx] ** 2))

    low_e  = band_energy(0, 300)      # 저음역
    mid_e  = band_energy(300, 3000)   # 중음역 (포만트)
    high_e = band_energy(3000, 8000)  # 고음역 (자음)

    feat = np.array([rms, zcr, low_e, mid_e, high_e], dtype=np.float32)
    return feat / (np.linalg.norm(feat) + 1e-12), freqs, mag

def cosine_sim(a, b):
    """코사인 유사도: cos(theta) = (A · B) / (||A|| * ||B||)"""
    return float(np.dot(a, b) / ((np.linalg.norm(a) + 1e-12) * (np.linalg.norm(b) + 1e-12)))

# --- [비교 실험 1] 두 학생의 가상 음성 샘플 생성 (또는 실제 마이크 녹음 파일 사용) ---
sr = 16000
t = np.linspace(0, 1.5, int(sr * 1.5), endpoint=False)

# 학생 1: 차분한 톤의 '안녕하세요' (기본 주파수 140Hz)
wave1 = 0.5 * np.sin(2 * np.pi * 140 * t) + 0.3 * np.sin(2 * np.pi * 700 * t) + 0.2 * np.sin(2 * np.pi * 2500 * t)
# 학생 2: 밝고 높은 톤의 '안녕하세요' (기본 주파수 220Hz)
wave2 = 0.5 * np.sin(2 * np.pi * 220 * t) + 0.35 * np.sin(2 * np.pi * 750 * t) + 0.25 * np.sin(2 * np.pi * 2600 * t)

feat1, freqs1, mag1 = make_feature_from_wave(wave1, sr)
feat2, freqs2, mag2 = make_feature_from_wave(wave2, sr)

similarity = cosine_sim(feat1, feat2)
sim_percent = max(0.0, min(100.0, similarity * 100.0))

# --- 시각화: 두 학생의 파형 및 스펙트럼 비교 ---
plt.figure(figsize=(12, 5))

plt.subplot(2, 1, 1)
plt.plot(t[:300], wave1[:300], label='학생 1 파형 (140Hz 톤)', color='#4F46E5', lw=1.5)
plt.plot(t[:300], wave2[:300], label='학생 2 파형 (220Hz 톤)', color='#10B981', lw=1.5, alpha=0.8)
plt.title('두 학생의 음성 파형 겹쳐보기 (Waveform Comparison)')
plt.xlabel('시간 (초)'); plt.ylabel('진폭'); plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
categories = ['RMS(음량)', 'ZCR(자음)', 'Low(저음)', 'Mid(모음)', 'High(마찰음)']
x = np.arange(len(categories))
width = 0.35
plt.bar(x - width/2, feat1, width, label='학생 1 특징 벡터', color='#4F46E5')
plt.bar(x + width/2, feat2, width, label='학생 2 특징 벡터', color='#10B981')
plt.xticks(x, categories)
plt.title(f'5차원 소리 지문 비교 ➔ 코사인 유사도 일치율: {sim_percent:.1f}%')
plt.ylabel('정규화된 특징값'); plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('=' * 65)
print(f'🎯 [두 학생의 목소리 코사인 유사도]: {similarity:.4f} (일치율: {sim_percent:.1f}%)')
if sim_percent >= 90.0:
    print('🏆 [판정]: 동일 인물이거나 발음이 거의 일치합니다! (화자/단어 일치)')
elif sim_percent >= 70.0:
    print('👥 [판정]: 같은 단어이지만 목소리 톤(음색)이 서로 다른 두 학생입니다!')
else:
    print('❌ [판정]: 서로 다른 단어를 말했거나 목소리 특징이 완전히 다릅니다!')
print('=' * 65)


---
## 6. 🏫 [미니 프로젝트 2] 우리 반 스마트 음성 비서 만들기!
학생이 마이크로 말하면, AI가 음성을 텍스트로 바꾸고(STT), 질문을 이해하여 친절하게 목소리로 대답해주는(TTS) 나만의 스마트 비서를 완성해봅시다!

* 🍱 *"오늘 급식 뭐야?"* ➔ 학교 실제 오늘 급식 메뉴 음성 안내!
* 📅 *"시간표 알려줘"* ➔ 1~4교시 시간표 안내
* 💡 *"교실 불 켜줘"* ➔ 교실 IoT 가상 제어
* 🗣️ *기타 질문* ➔ 친절한 AI 안내 멘트

In [ ]:
# 우리 반 스마트 비서 데이터 연동
TODAY_DATA = {
    "급식": "오늘의 중식 메뉴는 혼합잡곡밥, 된장찌개, 주꾸미불고기, 고르곤졸라피자입니다. 맛있게 드세요!",
    "시간표": "오늘은 1교시 국어, 2교시 수학, 3교시 자습, 4교시 체육 수업이 있습니다.",
    "일정": "이번 주 금요일에는 중간고사 성적 확인이 예정되어 있습니다.",
    "인사": "안녕하세요! 우리 반의 인공지능 스마트 비서입니다. 무엇을 도와드릴까요?"
}

def smart_classroom_assistant(voice_audio_path):
    # 1. 음성을 글자로 인식 (STT)
    result = model.transcribe(voice_audio_path, language="ko")
    command = result['text'].strip()
    print(f"👂 [학생의 음성 명령]: "{command}"")
    
    # 2. 인공지능의 의도 분석 (의도 분류)
    answer = ""
    if "급식" in command or "밥" in command or "점심" in command:
        answer = TODAY_DATA["급식"]
    elif "시간표" in command or "수업" in command:
        answer = TODAY_DATA["시간표"]
    elif "일정" in command or "행사" in command:
        answer = TODAY_DATA["일정"]
    elif "안녕" in command:
        answer = TODAY_DATA["인사"]
    elif "불" in command and ("켜" in command or "틀" in command):
        answer = "교실 전등을 켰습니다. 환한 환경에서 즐겁게 공부하세요!"
    else:
        answer = f"죄송해요, '{command}'라는 말을 아직 배우지 못했어요. 급식이나 시간표를 물어봐주세요!"
        
    print(f"🤖 [AI 비서 스마트 비서의 답변]: {answer}")
    
    # 3. AI 답변을 음성(TTS)으로 합성하여 들려주기
    tts = gTTS(text=answer, lang='ko')
    tts.save("assistant_reply.mp3")
    return ipd.Audio("assistant_reply.mp3", autoplay=True)

# 시뮬레이션: 학생이 "오늘 급식 메뉴 알려줘"라고 말했을 때!
demo_voice = create_sample_audio("오늘 점심 급식 메뉴 알려줘", "demo_cmd.wav")
smart_classroom_assistant(demo_voice)


---
## 7. 🌐 [생각해보기] 음성인식 기술과 인공지능 윤리
음성인식 기술은 우리의 삶을 놀랍도록 편리하게 만들어 주지만, 동시에 우리가 생각해보아야 할 질문들이 있습니다.

1. **장애인을 위한 따뜻한 기술**: 청각 장애인을 위한 실시간 자막 안경, 말을 하기 어려운 분을 위한 맞춤형 목소리 생성
2. **보안과 딥페이크(Deepfake) 목소리 피싱**: 부모님이나 친구의 목소리를 3초만 녹음해서 똑같이 흉내 내는 보이스피싱 범죄
3. **우리의 다짐**: 우리는 이 강력한 AI 기술을 어떻게 바르고 안전하게 활용해야 할까요?